In [ ]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24118743/versions/2 -O ../data/Chesseract.zip
# !unzip ../data/Chesseract.zip -d ../data/Chesseract/
# !rm ../data/Chesseract.zip

# wget doesn't work
# download and place archive to training/data/
# !mkdir ../data/Chesseract
# !unzip "../data/24118743.zip" -d ../data/Chesseract/

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=12 * 8 * 8, hidden_size = 26, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc_input = nn.Linear(input_size, hidden_size)
        self.act = nn.ReLU()
        self.fc_output = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = self.fc_input(x)
        x = self.act(x)
        x = self.fc_output(x)
        return x

In [ ]:
hyperparams = {
    "num_epochs": 126,
    "t_max": 18,  # num_epochs = 7 * t_max: expansions at epochs 18, 54, 90
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc_output": 0.5},
    "pruning_thresholds": {"fc_output": 1},
    "plateau_threshold": 0.01,
    "min_epochs_between_expansions": 8,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 500,
    "weight_decay": 0,
    "diff_threshold": 0.05,
    "warmup_epochs": 5
}

In [ ]:
train_x = np.load("../data/Chesseract/train_x.npy")
train_y = np.load("../data/Chesseract/train_y.npy")
valid_x = np.load("../data/Chesseract/valid_x.npy")
valid_y = np.load("../data/Chesseract/valid_y.npy")
test_x = np.load('../data/Chesseract/test_x.npy')
test_y = np.load('../data/Chesseract/test_y.npy')


train_tensor_x = torch.tensor(train_x).float().view(-1, 12 * 8 * 8)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 12 * 8 * 8)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 12 * 8 * 8)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = max(1, int(finder_fraction * len(train_dataset)))
finder_dataset = Subset(train_dataset, range(n_finder))

bs = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=0, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=bs, shuffle=False)
val_loader = DataLoader(valid_dataset, batch_size=bs, shuffle=False, num_workers=0, generator=g)
test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=0, generator=g)

In [ ]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="chesseract",
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

In [ ]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams['learning_rate']
    if current_lr is None:
        current_lr = base_lr

    new_optimizer = optim.Adam(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams['weight_decay'],
    )

    for group in new_optimizer.param_groups:
        group['initial_lr'] = base_lr

    return new_optimizer

trained_model = train_ssonn(
    model, train_loader, val_loader, finder_loader,
    criterion, optimizer_factory, hyperparams, device, experiment, True,
)
sparse_model = trained_model

In [ ]:
experiment.end()

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
_, params = get_param_counts(sparse_model)  # (total, non-zero): report the live parameters

In [ ]:
accuracy, params

In [ ]:
torch.save(sparse_model, 'chesseract.pt')

In [ ]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 57.83, 57.45, 55.69, 52.74, 56.26, 59.60, 55.15, 57.20, 58.24, 60.76, 59.16, 68.83, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))
styled_table